In [14]:
from unet import unet_model
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = unet_model.UNet(
    n_channels=3,   # RGB-like input
    n_classes=1,    # scalar field output
    bilinear=False,
).to(device)


# One fake 64x64 RGB image
x = torch.rand(
    1, 3, 64, 64,
    device=device,
)

model.eval()

with torch.no_grad():
    output = model(x)


print("Device:", device)
print("Input: ", x.shape)
print("Output:", output.shape)

Device: cpu
Input:  torch.Size([1, 3, 64, 64])
Output: torch.Size([1, 1, 64, 64])


In [16]:
from unet.hexunet_parts import HDoubleConv
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

class HexCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = HDoubleConv(1, 8)
        self.conv2 = HDoubleConv(8, 16)
        self.conv3 = HDoubleConv(16, 1)

    def forward(self, x):

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)

        return x

In [17]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = HexCNN().to(device)

x = torch.rand(
    1,      # batch
    1,      # channel
    64, 64, # axial q/r storage
    device=device,
)

with torch.no_grad():
    y = model(x)

print("Input: ", x.shape)
print("Output:", y.shape)

Input:  torch.Size([1, 1, 64, 64])
Output: torch.Size([1, 1, 64, 64])


In [7]:
from torch.utils.data import TensorDataset, DataLoader


def generate_hex_dataset(
    n_samples=1000,
    height=64,
    width=64,
):
    X = torch.rand(
        n_samples,
        1,
        height,
        width,
    )

    kernel = torch.tensor([
        [0., 1., 1.],
        [1., 1., 1.],
        [1., 1., 0.],
    ]).reshape(1, 1, 3, 3)

    y = F.conv2d(
        X,
        kernel,
        padding=1,
    )

    return X, y


X, y = generate_hex_dataset()

dataset = TensorDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)


model = HexCNN()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

loss_fn = torch.nn.MSELoss()


for epoch in range(20):

    total_loss = 0.0

    model.train()

    for X_batch, y_batch in loader:

        pred = model(X_batch)

        loss = loss_fn(
            pred,
            y_batch,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    print(
        epoch,
        average_loss,
    )


model.eval()

with torch.no_grad():
    pred = model(X[:1])


print("Actual:")
print(
    y[0, 0, 30:35, 30:35]
)

print("Predicted:")
print(
    pred[0, 0, 30:35, 30:35]
)

0 9.689681500196457
1 9.106701344251633
2 8.69916769862175
3 8.430621564388275
4 8.200813055038452
5 7.977484911680222
6 7.760104209184647
7 7.543635532259941
8 7.32993483543396
9 7.126199036836624
10 6.934148997068405
11 6.7508063316345215
12 6.571417659521103
13 6.394187092781067
14 6.224709257483482
15 6.05192144215107
16 5.881527841091156
17 5.719834074378014
18 5.55351759493351
19 5.394129827618599
Actual:
tensor([[3.8533, 4.5248, 4.5160, 4.6633, 4.6476],
        [4.0423, 3.6902, 4.2947, 4.0369, 2.5836],
        [3.7361, 4.3877, 3.5739, 4.0355, 2.1976],
        [3.3145, 3.8956, 3.5158, 2.3857, 2.7155],
        [3.3537, 2.8490, 3.2860, 3.2791, 2.5316]])
Predicted:
tensor([[1.6534, 2.0694, 2.0999, 2.1270, 2.1322],
        [1.7082, 1.5770, 1.8527, 1.8153, 0.0000],
        [1.6379, 2.0040, 1.5884, 1.8068, 0.0000],
        [1.2959, 1.7594, 1.6410, 0.0000, 0.0000],
        [1.5239, 0.0000, 1.2598, 1.4978, 0.0000]])


In [8]:
X

tensor([[[[0.0241, 0.8277, 0.6966,  ..., 0.5351, 0.7366, 0.6199],
          [0.1441, 0.6812, 0.0119,  ..., 0.2159, 0.9004, 0.8962],
          [0.3342, 0.4385, 0.9143,  ..., 0.7159, 0.6391, 0.0139],
          ...,
          [0.5054, 0.8274, 0.2496,  ..., 0.6824, 0.0667, 0.6105],
          [0.8000, 0.8463, 0.6264,  ..., 0.3362, 0.3851, 0.1082],
          [0.1951, 0.3586, 0.6563,  ..., 0.9095, 0.0836, 0.5063]]],


        [[[0.8630, 0.2148, 0.4284,  ..., 0.4929, 0.0137, 0.1484],
          [0.3173, 0.7774, 0.8939,  ..., 0.7162, 0.2415, 0.6611],
          [0.5713, 0.6543, 0.5434,  ..., 0.9438, 0.5975, 0.6074],
          ...,
          [0.5412, 0.9039, 0.2685,  ..., 0.7937, 0.6001, 0.2931],
          [0.6577, 0.7581, 0.2797,  ..., 0.0175, 0.5813, 0.7338],
          [0.1057, 0.5878, 0.3065,  ..., 0.6824, 0.8852, 0.7783]]],


        [[[0.7334, 0.0347, 0.1790,  ..., 0.2960, 0.5585, 0.0767],
          [0.9026, 0.3687, 0.0695,  ..., 0.5017, 0.2091, 0.9261],
          [0.9374, 0.4686, 0.1319,  ..

In [9]:
import torch
import torch.nn.functional as F


size = 7
center = 3

x = torch.zeros(
    1,
    1,
    size,
    size,
)

x[0, 0, center, center] = 1.0


kernel = torch.tensor([
    [0., 1., 0.],
    [1., 1., 1.],
    [0., 1., 0.],
]).reshape(1, 1, 3, 3)


y = F.conv2d(
    x,
    kernel,
    padding=1,
)


print("Input:")
print(x[0, 0])

print("\nOutput:")
print(y[0, 0])


active = torch.nonzero(
    y[0, 0],
    as_tuple=False,
)

print("\nActive output cells:")
print(active)

Input:
tensor([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]])

Output:
tensor([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 1., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]])

Active output cells:
tensor([[2, 3],
        [3, 2],
        [3, 3],
        [3, 4],
        [4, 3]])
